# Récupération des données des log du Branch and Bound beta-CROWN

In [1]:
import re
import pandas as pd

def parse_verification_report(path):
    """
    Lit un fichier texte contenant un rapport au format donné
    et extrait les informations importantes dans un dictionnaire.
    """

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    data = {}

    # Final verified acc
    m = re.search(r"Final verified acc:\s*([\d\.]+)%", text)
    if m:
        data["final_verified_accuracy"] = float(m.group(1))

    # Problem instances count
    m = re.search(r"Problem instances count:\s*(\d+)", text)
    if m:
        data["problem_instances_count"] = int(m.group(1))

    # total verified, total falsified, timeout
    m = re.search(
        r"total verified.*?:\s*(\d+)\s*,\s*total falsified.*?:\s*(\d+)\s*,\s*timeout:\s*(\d+)",
        text
    )
    if m:
        data["verified"] = int(m.group(1))
        data["falsified"] = int(m.group(2))
        data["timeout"] = int(m.group(3))

    # mean / max time for all instances
    m = re.search(
        r"mean time for ALL instances.*?:([\d\.]+).*?max time:\s*([\d\.]+)",
        text
    )
    if m:
        data["mean_time_all"] = float(m.group(1))
        data["max_time_all"] = float(m.group(2))

    # mean / max time for verified SAFE
    m = re.search(
        r"mean time for verified SAFE instances.*?:\s*([\d\.]+).*?max time:\s*([\d\.]+)",
        text
    )
    if m:
        data["mean_time_safe"] = float(m.group(1))
        data["max_time_safe"] = float(m.group(2))

    # safe-incomplete list
    m = re.search(r"safe-incomplete.*?index:\s*\[(.*?)\]", text)
    if m:
        indices = m.group(1).strip()
        data["safe_incomplete_indices"] = (
            [] if indices == "" else [int(x) for x in indices.split(",")]
        )

    return data


In [2]:
result = parse_verification_report("/share/homes/boyerma/FastSDPCertification/results/benchmark/6x100-0.026/Branch-and-Bound-beta-CROWN/data_index=0-71.txt")
print(result)


{'final_verified_accuracy': 97.22222222222221, 'problem_instances_count': 72, 'verified': 70, 'falsified': 2, 'timeout': 0, 'mean_time_all': 0.5151906059561167, 'max_time_all': 8.080828666687012, 'mean_time_safe': 0.5295758826392037, 'max_time_safe': 8.080828666687012, 'safe_incomplete_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 71]}


In [3]:
import os
import sys

folder = "/share/homes/boyerma/FastSDPCertification/results/benchmark/6x100-0.026/Branch-and-Bound-beta-CROWN/" 
results = pd.DataFrame(columns=[
    "filename","final_verified_accuracy","problem_instances_count",
    "verified","falsified","timeout",
    "mean_time_all","max_time_all",
    "mean_time_safe","max_time_safe",
    "safe_incomplete_indices"
])

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        path = os.path.join(folder, filename)
        result = parse_verification_report(path)
        print(f"Results for {filename}:")
        print(result)
        results = pd.concat([results, pd.DataFrame([{"filename": filename, **result}])], ignore_index=True)
        print("-" * 40)

Results for data_index=73-76.txt:
{'final_verified_accuracy': 75.0, 'problem_instances_count': 4, 'verified': 3, 'falsified': 1, 'timeout': 0, 'mean_time_all': 0.6005850320838676, 'max_time_all': 1.2320044040679932, 'mean_time_safe': 0.6502602895100912, 'max_time_safe': 1.2320044040679932, 'safe_incomplete_indices': [1, 2, 3]}
----------------------------------------
Results for data_index=0-71.txt:
{'final_verified_accuracy': 97.22222222222221, 'problem_instances_count': 72, 'verified': 70, 'falsified': 2, 'timeout': 0, 'mean_time_all': 0.5151906059561167, 'max_time_all': 8.080828666687012, 'mean_time_safe': 0.5295758826392037, 'max_time_safe': 8.080828666687012, 'safe_incomplete_indices': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 71]}
------------------------------------

/tmp/ipykernel_1875/2815720875.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([{"filename": filename, **result}])], ignore_index=True)


In [4]:
results["total"] = results["verified"] + results["falsified"] + results["timeout"]

In [7]:
results["verified"].sum()

95

In [8]:
results["falsified"].sum()

5

In [6]:
results.head(20)

,filename,final_verified_accuracy,problem_instances_count,verified,falsified,timeout,mean_time_all,max_time_all,mean_time_safe,max_time_safe,safe_incomplete_indices,total
0,data_index=73-76.txt,75.000000,4,3,1,0,0.600585,1.232004,0.650260,1.232004,"[1, 2, 3]",4
1,data_index=0-71.txt,97.222222,72,70,2,0,0.515191,8.080829,0.529576,8.080829,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",72
2,data_index=79-80.txt,50.000000,2,1,1,0,0.501778,0.991666,0.991666,0.991666,[0],2
3,data_index=82-86.txt,100.000000,5,5,0,0,0.436388,0.998864,0.436389,0.998864,"[0, 1, 2, 3, 4]",5
4,data_index=89-91.txt,100.000000,3,3,0,0,0.557044,1.064052,0.557046,1.064052,"[0, 1, 2]",3
5,data_index=94.txt,100.000000,1,1,0,0,1.044091,1.044102,1.044102,1.044102,[0],1
6,data_index=96-97.txt,100.000000,2,2,0,0,0.676838,1.064196,0.676842,1.064196,"[0, 1]",2
7,data_index=100.txt,100.000000,1,1,0,0,1.212509,1.212522,1.212522,1.212522,[0],1
8,data_index=109.txt,100.000000,1,1,0,0,1.035812,1.035822,1.035822,1.035822,[0],1
9,data_index=117.txt,100.000000,1,1,0,0,1.050426,1.050437,1.050437,1.050437,[0],1


In [9]:
results["total_time"] = results["mean_time_all"] * results["total"]

In [10]:
results["total_time"].mean()

3.335106439330147

# Verification output

In [1]:
import pickle

with open('/share/homes/boyerma/alpha-beta-CROWN/complete_verifier/verification_output.pkl', 'rb') as f:
    verification_output = pickle.load(f)
    


/share/homes/boyerma/miniconda3/envs/share_envpy311/lib/python3.9/site-packages/torch/package/_mock_zipreader.py:17: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at  /opt/conda/conda-bld/pytorch_1631630778054/work/torch/csrc/utils/tensor_numpy.cpp:67.)
  _dtype_to_storage = {data_type(0).dtype: data_type for data_type in _storages}


In [2]:
verification_output

{'idx': 0,
 'pred': None,
 'attack_margin': None,
 'pred_adv': None,
 'init_crown_bounds': None,
 'init_alpha_crown': None,
 'refined_lb': None,
 'decisions': [],
 'results': 'unsafe-pgd',
 'time': 0.6478345394134521,
 'domains_visited': None,
 'selected_dataset_raw': {'X': tensor([[[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000],
            [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000],
            [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
             0.0000

# Vérification des exemples adverses

In [4]:
#!/usr/bin/env python3
"""
Script to extract X and Y tensors from counterexample file.
Reads the cex_path file and extracts:
- X tensor: 784 values (X_0 to X_783)
- Y tensor: 10 values (Y_0 to Y_9)
"""

import re
import numpy as np
from pathlib import Path


def extract_tensors_from_cex(cex_path):
    """
    Extract X and Y tensors from counterexample file.
    
    Args:
        cex_path (str): Path to the counterexample file
        
    Returns:
        tuple: (X_tensor, Y_tensor) as numpy arrays
    """
    X_values = {}
    Y_values = {}
    
    with open(cex_path, 'r') as f:
        content = f.read()
    
    # Parse all (name value) pairs
    pattern = r'\((\w+_\d+)\s+([-\d.]+)\)'
    matches = re.findall(pattern, content)
    
    for name, value in matches:
        value = float(value)
        
        if name.startswith('X_'):
            idx = int(name.split('_')[1])
            X_values[idx] = value
        elif name.startswith('Y_'):
            idx = int(name.split('_')[1])
            Y_values[idx] = value
    
    # Convert to numpy arrays (sorted by index)
    X_tensor = np.array([X_values[i] for i in sorted(X_values.keys())])
    Y_tensor = np.array([Y_values[i] for i in sorted(Y_values.keys())])
    
    return X_tensor, Y_tensor


In [5]:
"""Main function."""
cex_path = '/share/homes/boyerma/alpha-beta-CROWN/complete_verifier/test_margot.txt'

# Check if file exists
if not Path(cex_path).exists():
    print(f"Error: File '{cex_path}' not found")
    exit()
    
# Extract tensors
X_adv, Y_adv = extract_tensors_from_cex(cex_path)

print(f"Successfully extracted tensors from '{cex_path}'")
print(f"\nX tensor shape: {X_adv.shape}")
print(f"X tensor min: {X_adv.min():.6f}, max: {X_adv.max():.6f}, mean: {X_adv.mean():.6f}")
print(f"X tensor first 10 values: {X_adv[:10]}")
print(f"X tensor last 10 values: {X_adv[-10:]}")

print(f"\nY tensor shape: {Y_adv.shape}")
print(f"Y tensor values:\n{Y_adv}") 
# Optional: save to files
np.save('X_tensor.npy', X_adv)
np.save('Y_tensor.npy', Y_adv)
print(f"\nTensors saved to 'X_tensor.npy' and 'Y_tensor.npy'")

# Optional: save as text files
np.savetxt('X_tensor.txt', X_adv, fmt='%.6f')
np.savetxt('Y_tensor.txt', Y_adv, fmt='%.6f')
print(f"Tensors also saved to 'X_tensor.txt' and 'Y_tensor.txt'")



Successfully extracted tensors from '/share/homes/boyerma/alpha-beta-CROWN/complete_verifier/test_margot.txt'

X tensor shape: (784,)
X tensor min: 0.000000, max: 0.974000, mean: 0.113157
X tensor first 10 values: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
X tensor last 10 values: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

Y tensor shape: (10,)
Y tensor values:
[-10.21920776  -7.33659267  -3.67729521   1.17926931   8.22635174
   0.35640112  -6.07050467   2.92742085   2.75901294   6.85418177]

Tensors saved to 'X_tensor.npy' and 'Y_tensor.npy'
Tensors also saved to 'X_tensor.txt' and 'Y_tensor.txt'


In [6]:
from data import load_dataset
from tools import get_project_path
from torch.utils.data import DataLoader
from networks import ReLUNN

dataset = load_dataset(get_project_path(f"config/mnist-6x100.yaml"))
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)


for i, (x, ytrue) in enumerate(dataloader):
    if i not in [97]:
    
        continue
    else :
        print(f"Example {i}:")
        print(f"x: {x}")
        print(f"ytrue: {ytrue}")

        diff = x.flatten() - X_adv
        print(f"diff min : {diff.min():.6f}, max: {diff.max():.6f}, mean: {diff.mean():.6f}")


network = ReLUNN.from_yaml(get_project_path("config/mnist-6x100.yaml"))

config :  {'data': {'name': 'mnist', 'path': 'data/datasets/mnist_subset_10_per_class.pth', 'num_classes': 10, 'num_samples': 100}, 'input_ball': {'norm': 'Linf', 'epsilon': 0.026}, 'network': {'name': '6x100', 'path': 'data/models/mnist_adv_6x100.pt', 'K': 7, 'n': [784, 100, 100, 100, 100, 100, 100, 10]}, 'models': [{'certification_model_name': 'MdSDP', 'cuts': ['RLT', 'triangularization', 'Tij', 'beta_logits_comparaison_1', 'beta_logits_comparaison_2', 'McC_betaz_logits', 'Tij_before_penultimate_layer'], 'RLT_props': [1.0], 'all_combinations_cuts': False, 'MATRIX_BY_LAYERS': True, 'LAST_LAYER': False, 'use_fusion': False, 'use_callback': False, 'use_active_neurons': False, 'use_inactive_neurons': False, 'keep_penultimate_actives': True, 'bounds_method': 'alpha-CROWN', 'alpha_1': 0.5, 'alpha_2': 0.5}]}
Example 97:
x: tensor([[[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0

/tmp/ipykernel_72133/2074502292.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diff = x.flatten() - X_adv


OrderedDict([('layers.Layer_1_Linear.weight', tensor([[ 1.9806e-04,  1.3185e-04,  2.5926e-04,  ...,  3.8802e-05,
          5.3384e-04,  7.9886e-05],
        [ 1.4599e-04, -4.4716e-04,  1.7912e-04,  ...,  4.3904e-04,
          5.4067e-04, -1.9919e-04],
        [-1.2300e-02, -1.0815e-02, -1.1824e-02,  ..., -9.2488e-03,
         -6.1684e-03, -8.5918e-03],
        ...,
        [ 1.2391e-04,  3.5267e-04,  2.8746e-05,  ...,  8.8004e-06,
         -5.0452e-06, -3.0167e-04],
        [-8.5115e-04, -5.4688e-04, -6.6215e-04,  ..., -7.4446e-04,
         -9.9992e-04, -3.6547e-04],
        [-1.5779e-04, -2.2351e-04, -5.0435e-04,  ..., -3.1646e-04,
         -3.0491e-04, -2.1600e-04]], device='cuda:0')), ('layers.Layer_1_Linear.bias', tensor([ 8.4779e-02, -3.1400e-01,  2.8114e-02, -1.1611e-01,  1.2482e-01,
        -1.0199e-01,  5.6465e-01, -6.9030e-02,  1.0881e-01, -6.1329e-02,
         4.8022e-02,  2.5972e-01,  3.6517e-02, -9.7297e-02, -4.3470e-01,
        -2.6251e-01,  2.1883e-01, -4.9770e-01,  1.249

In [16]:
import torch

X_adv = torch.tensor(X_adv.reshape(1,784), dtype = torch.float)

network.forward(X_adv)

/tmp/ipykernel_72133/1594471891.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_adv = torch.tensor(X_adv.reshape(1,784), dtype = torch.float)


tensor([[-10.2192,  -7.3366,  -3.6773,   1.1793,   8.2264,   0.3564,  -6.0705,
           2.9274,   2.7590,   6.8542]])